# 📊 Módulo 5 (visual) — Comparar experimentos do MLflow

Acompanha o **Módulo 5** do guia (`docs/02_GUIA_DE_APRENDIZADO.md`).
Em vez de só abrir a UI do MLflow, aqui lemos os experimentos **via API** e
montamos nossos próprios gráficos e tabelas — que é o que um engenheiro de
MLOps faz para relatórios e automações.

> Abra com `make lab` a partir da raiz, com o `.venv` ativo.

## 1. Setup e conexão com o backend local do MLflow

Nossos treinos gravam em `./mlruns` (backend em arquivo). Apontamos o MLflow
para lá.

In [ ]:
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import mlflow

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)
# MLflow 3.x usa backend em banco; usamos SQLite local (mesmo dos scripts).
mlflow.set_tracking_uri(f"sqlite:///{os.path.join(ROOT, 'mlflow.db')}")
print("Tracking URI:", mlflow.get_tracking_uri())

## 2. Garantir que existem experimentos para comparar

Se você ainda não rodou nada, executamos a demo (`mlops/mlflow_demo.py`), que
cria 3 execuções com *learning rates* diferentes.

In [ ]:
exps = [e.name for e in mlflow.search_experiments()]
print("Experimentos encontrados:", exps)
if "demo-mlops" not in exps:
    print("Rodando a demo para gerar runs...")
    subprocess.run([sys.executable, "mlops/mlflow_demo.py"], check=True)
    print("Demo concluída.")

## 3. Ler as execuções como tabela (pandas)

`mlflow.search_runs` devolve um DataFrame — perfeito para inspecionar params e
métricas lado a lado.

In [ ]:
df = mlflow.search_runs(experiment_names=["demo-mlops"])
cols = [c for c in df.columns
        if c.startswith("params.") or c.startswith("metrics.") or c == "run_id"]
df[cols].head(10)

## 4. 📈 Comparar as curvas de loss de cada run

Aqui está o poder do tracking: sobrepor as curvas de execuções diferentes para
decidir qual hiperparâmetro foi melhor. Puxamos o histórico completo da métrica
`loss` de cada run via `MlflowClient`.

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
plt.figure(figsize=(9, 5))
for _, row in df.iterrows():
    run_id = row["run_id"]
    lr = row.get("params.learning_rate", "?")
    hist = client.get_metric_history(run_id, "loss")
    hist = sorted(hist, key=lambda m: m.step)
    steps = [m.step for m in hist]
    losses = [m.value for m in hist]
    if steps:
        plt.plot(steps, losses, label=f"lr={lr}")

plt.title("Comparação de runs — loss por passo")
plt.xlabel("passo")
plt.ylabel("loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 📊 Gráfico de barras: loss final por configuração

Uma visão resumida: qual configuração terminou com a menor loss?

In [ ]:
labels, finais = [], []
for _, row in df.iterrows():
    lr = row.get("params.learning_rate", "?")
    hist = sorted(client.get_metric_history(row["run_id"], "loss"),
                  key=lambda m: m.step)
    if hist:
        labels.append(f"lr={lr}")
        finais.append(hist[-1].value)

plt.figure(figsize=(7, 4))
bars = plt.bar(labels, finais, color="steelblue")
plt.bar_label(bars, fmt="%.3f")
plt.title("Loss final por configuração (menor é melhor)")
plt.ylabel("loss final")
plt.tight_layout()
plt.show()

## 6. E o treino real? (experimento `mini-gpt-from-scratch`)

O script `02_train_tiny_gpt.py` também registra no MLflow. Se você já o rodou,
este experimento aparece aqui — mesma técnica de leitura.

In [ ]:
if "mini-gpt-from-scratch" in [e.name for e in mlflow.search_experiments()]:
    dfg = mlflow.search_runs(experiment_names=["mini-gpt-from-scratch"])
    print(f"{len(dfg)} run(s) do mini-GPT encontradas.")
    if len(dfg):
        rid = dfg.iloc[0]["run_id"]
        for metric in ["train_loss", "val_loss"]:
            hist = sorted(client.get_metric_history(rid, metric),
                          key=lambda m: m.step)
            if hist:
                plt.plot([m.step for m in hist], [m.value for m in hist],
                         label=metric, marker=".")
        plt.title("mini-GPT from scratch — última run")
        plt.xlabel("passo"); plt.ylabel("loss"); plt.legend()
        plt.grid(True, alpha=0.3); plt.show()
else:
    print("Ainda não há runs do mini-GPT. Rode 'make train-tiny' e volte aqui.")

## 7. 🧪 Exercícios

1. Rode `make train-tiny` com learning rates diferentes (`--lr 1e-3` e `--lr 1e-2`)
   e volte para comparar as curvas do experimento `mini-gpt-from-scratch`.
2. Abra a UI oficial com `make mlflow` (http://127.0.0.1:5000) e confira que os
   mesmos dados aparecem lá.
3. Adicione uma métrica nova nos seus scripts (ex.: `mlflow.log_metric("lr", ...)`)
   e plote-a aqui.

📝 Registre suas conclusões no `docs/03_DIARIO.md`.